In [1]:
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from copy import deepcopy
from string import ascii_lowercase

parent = os.path.abspath("..")
if parent not in sys.path:
    sys.path.append(parent)

from src.modeling import translation_model


In [4]:
net = deepcopy(translation_model.net)
optimizer = deepcopy(translation_model.optimizer)
transform = deepcopy(translation_model.transform)
BATCH_SIZE = translation_model.BATCH_SIZE
WORKERS = translation_model.WORKERS

In [8]:
save_path = "../models/runs/translation/train4/best_model_epoch92.pth"

checkpoint = torch.load(save_path, weights_only=True)
net.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
epoch = checkpoint["epoch"]
loss = checkpoint["loss"]

text_accuracy = (
    f" y exactitud {checkpoint['accuracy']:.4%}" if "accuracy" in checkpoint else ""
)

print(f"Cargado modelo de la época {epoch}, con pérdida {loss:.4f}{text_accuracy} --> Entrenamiento")

Cargado modelo de la época 92, con pérdida 2.3221 y exactitud 100.0000% --> Entrenamiento


In [9]:
class CustomDataset(Dataset):
    ALLOWED_EXTENSIONS: list[str] = [".jpg", ".jpeg", ".png"]

    def __init__(self, root: str, transform: transforms.Compose | None = None) -> None:
        if not os.path.exists(root):
            raise FileNotFoundError(f"Path {root} does not exist")

        self.root: Path = Path(root)
        self.transform = transform

        images_path = [
            list(self.root.rglob(f"*{ext}")) for ext in self.ALLOWED_EXTENSIONS
        ]

        self.imgs_path: list[Path] = [
            item for sublist in images_path for item in sublist
        ]

    def __len__(self) -> int:
        return len(self.imgs)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_path = self.imgs_path[index]

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32) / 255.0  # Normaliza a [0, 1]

        if self.transform:
            img = self.transform(img)
        else:
            img = torch.from_numpy(img).float()

        # Ignorar el segundo elemento de la tupla
        label = torch.from_numpy(np.uint16([index]))

        # if ngpus > 0:
        #     img = torch.cuda.FloatTensor(img)
        #     label = label.to(device)

        return img, label

    @property
    def imgs(self) -> list[Path]:
        return self.imgs_path

In [10]:
custom_dataset = CustomDataset(
    root="../data/processed/character",
    transform=transform,
)

dataloader = DataLoader(
    custom_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS
)

In [11]:
rows, columns = 20, 257
fig, axes = plt.subplots(rows, columns, figsize=(60, 10))
axes_flatten = axes.flatten()

cont = 0
with torch.no_grad():
    for images, _ in dataloader:
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)

        images = images.numpy().reshape(
            images.shape[0], images.shape[2], images.shape[3]
        )  # (batch_size, height, width)
        for i in range(images.shape[0]):
            img = images[i, :, :]

            ax = axes_flatten[cont]

            character_predicted = ascii_lowercase[predicted[i]]
            ax.set(xticks=[], yticks=[])
            ax.set_title(character_predicted, fontsize=3, pad=1)

            scale = 10
            img = cv2.resize(
                img, (img.shape[1] * scale, img.shape[0] * scale), interpolation=cv2.INTER_CUBIC
            )
            ax.imshow(img, cmap="gray")
            cont += 1

fig.savefig("../models/runs/translation/train4/predictions.svg", format="svg", dpi=1000)
plt.close()